# **Projet 4 - Analyse des causes d'attrition au sein d'une ESN**

> - Réaliser une analyse exploratoire des données afin d'identifier les principales tendances et différences entre les employés.
> - Identifier les facteurs potentiellement associés à l'attrition des employés.
> - Préparer et nettoyer les données afin de construire un jeu de données centralisé et exploitable pour la modélisation.
> - Comparer plusieurs modèles de classification supervisée afin de prédire le risque d'attrition.
> - Identifier les variables ayant le plus d'influence sur l'attrition des employés.

## **Contexte**

> -  Une entreprise de services du numérique (ESN) souhaite mieux comprendre les facteurs associés au départ de ses employés.
> - L'entreprise dispose de plusieurs fichiers de données contenant des informations sur les employés, leur situation professionnelle et leurs caractéristiques.
> - L'objectif est d'explorer et de croiser ces données afin d'identifier les principales tendances associées à l'attrition et de préparer les données pour la modélisation prédictive.

 ## **Objectif**
> - Identifier les principaux facteurs associés à l'attrition des employés afin de mieux comprendre les causes de départ au sein de l'ESN et de construire un modèle permettant de prédire le risque d'attrition.

## **Objectifs Data**

> - **E1 - Réaliser une analyse exploratoire des fichiers de données :** analyser la structure des données, identifier les distributions, les valeurs manquantes, les doublons et les différences entre les employés ayant quitté l'entreprise et ceux qui sont restés.
> - **E2 - Préparer la donnée pour la modélisation :** nettoyer les données, traiter les incohérences, réaliser les jointures nécessaires et préparer les variables pour la modélisation.
> - **E3 - Réaliser un premier modèle de classification :** préparer les jeux d'apprentissage et de test, entraîner plusieurs modèles de classification et évaluer leurs premières performances.
> - **E4 - Améliorer l'approche de classification :** comparer les modèles, améliorer les performances et sélectionner l'approche de classification la plus pertinente.
> - **E5 - Optimiser et interpréter le modèle :** optimiser les hyperparamètres du modèle retenu, évaluer ses performances et identifier les variables ayant le plus d'influence sur l'attrition à l'aide de SHAP.
> - **E6 - Formaliser les résultats :** synthétiser les principaux enseignements de l'analyse et formuler des recommandations actionnables pour réduire l'attrition.

## **ÉTAPE 3 - Premier modèle de classification**

### **Importation des librairies**

In [1]:
# Importation des librairies

import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Fonctions du module src

import os
import sys

print("Importation des librairies OK")

Importation des librairies OK


In [2]:
# ============================================================
# IMPORTATION DES LIBRAIRIES - E3 CLASSIFICATION
# ============================================================

# Bibliothèques générales
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Séparation / validation / sélection des modèles
# ============================================================

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_validate,
    StratifiedKFold
)

# ============================================================
# Métriques
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)

# ============================================================
# Préprocessing
# ============================================================

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler
)

# ============================================================
# Modèles
# ============================================================

from sklearn.dummy import DummyClassifier

from sklearn.linear_model import (
    LogisticRegression
)

from sklearn.ensemble import (
    RandomForestClassifier
)

# ============================================================
# Pipeline
# ============================================================

from sklearn.pipeline import Pipeline

# ============================================================
# Configuration du chemin vers le projet
# ============================================================

current_path = Path.cwd()

for path in [current_path] + list(current_path.parents):
    if (path / "src").is_dir():
        PROJECT_ROOT = path
        break
else:
    raise FileNotFoundError(
        "Le dossier 'src' n'a pas été trouvé."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# Fonctions du module src
# ============================================================

from src.feature_engineering import prepare_modelisation_data

# ============================================================
# Vérification
# ============================================================

print("Importation des librairies OK")
print("Module src importé depuis :", PROJECT_ROOT / "src")

Importation des librairies OK
Module src importé depuis : c:\Users\FR103217\OneDrive - MOTUL\Bureau\data_scientist\project_4\src


### **Chargement des jeux des données**

In [3]:
# Importation du fichier central

df_modelisation = pd.read_csv("../data/processed/df_modelisation.csv", index_col=0)

print("Importation du fichier central OK")
print(f"Dimensions du dataset : {df_modelisation.shape[0]} lignes, {df_modelisation.shape[1]} colonnes")

display(df_modelisation.head())

Importation du fichier central OK
Dimensions du dataset : 1470 lignes, 46 colonnes


,nominal__genre_M,nominal__statut_marital_Célibataire,nominal__statut_marital_Divorcé(e),nominal__statut_marital_Marié(e),nominal__departement_Commercial,nominal__departement_Consulting,nominal__departement_Ressources Humaines,nominal__poste_Assistant de Direction,nominal__poste_Cadre Commercial,nominal__poste_Consultant,...,remainder__satisfaction_employee_equipe,remainder__satisfaction_employee_equilibre_pro_perso,remainder__note_evaluation_actuelle,remainder__augementation_salaire_precedente,remainder__nombre_participation_pee,remainder__nb_formations_suivies,remainder__distance_domicile_travail,remainder__niveau_education,remainder__annees_depuis_la_derniere_promotion,a_quitte_l_entreprise
nominal__genre_F,,,,,,,,,,,,,,,,,,,,,
1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,3.0,11.0,0.0,0.0,1.0,2.0,0.0,Oui
0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,4.0,3.0,4.0,23.0,1.0,3.0,8.0,1.0,1.0,Non
0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,2.0,3.0,3.0,15.0,0.0,3.0,2.0,2.0,0.0,Oui
1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,3.0,3.0,3.0,11.0,0.0,3.0,3.0,4.0,3.0,Non
0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,4.0,3.0,3.0,12.0,1.0,3.0,2.0,1.0,2.0,Non


### **Séparation des données (train/test)**

#### **Définition de X et y**

Commencer par réaliser une séparation train test simple ou une validation croisée simple.

In [4]:
# Définition des variables explicatives et de la variable cible

X = df_modelisation.drop("a_quitte_l_entreprise", axis=1)
y = df_modelisation["a_quitte_l_entreprise"]

print(f"Dimensions de X : {X.shape}")
print(f"Dimensions de y : {y.shape}")

Dimensions de X : (1470, 45)
Dimensions de y : (1470,)


#### **Séparation des jeux d'apprentissage et de test**

In [5]:
# Séparation des données en jeu d'entraînement et jeu de test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)
print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test :", y_test.shape)

Dimensions de X_train : (1176, 45)
Dimensions de X_test : (294, 45)
Dimensions de y_train : (1176,)
Dimensions de y_test : (294,)


In [6]:
# Vérification de la répartition de la variable cible

print("Répartition de la cible dans le jeu complet :")
print(y.value_counts(normalize=True).mul(100).round(2))

print("\nRépartition dans le jeu d'entraînement :")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nRépartition dans le jeu de test :")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Répartition de la cible dans le jeu complet :
a_quitte_l_entreprise
Non    83.88
Oui    16.12
Name: proportion, dtype: float64

Répartition dans le jeu d'entraînement :
a_quitte_l_entreprise
Non    83.84
Oui    16.16
Name: proportion, dtype: float64

Répartition dans le jeu de test :
a_quitte_l_entreprise
Non    84.01
Oui    15.99
Name: proportion, dtype: float64


### **Modèle Dummy**

> Le modèle **Dummy** sert de référence (benchmark) pour établir une performance minimale. Nous utilisons la stratégie `most_frequent`, qui prédit systématiquement la classe la plus fréquente du jeu d'apprentissage. Les modèles suivants seront comparés à cette référence afin d'évaluer leur réelle capacité de classification.

In [7]:
# Modèle étalon - DummyClassifier

dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

dummy_model.fit(X_train, y_train)

print("Entraînement du modèle Dummy OK")

Entraînement du modèle Dummy OK


#### **Prédictions du modèle Dummy**

In [8]:
# Prédictions du modèle Dummy

y_train_pred_dummy = dummy_model.predict(X_train)
y_test_pred_dummy = dummy_model.predict(X_test)

print("Prédictions du modèle Dummy OK")

Prédictions du modèle Dummy OK


#### **Évaluation - jeu d’apprentissage**

In [9]:
# Évaluation du modèle Dummy - Jeu d'apprentissage

print("=== Dummy - Jeu d'apprentissage ===")

print(f"Accuracy : {accuracy_score(y_train, y_train_pred_dummy):.3f}")
print(f"Precision : {precision_score(y_train, y_train_pred_dummy, pos_label="Oui", zero_division=0):.3f}")
print(f"Recall : {recall_score(y_train, y_train_pred_dummy, pos_label="Oui", zero_division=0):.3f}")

print("\nMatrice de confusion :")
print(confusion_matrix(y_train, y_train_pred_dummy, labels=["Non", "Oui"]))

print("\nClassification report :")
print(classification_report(y_train, y_train_pred_dummy, labels=["Non", "Oui"], zero_division=0))

=== Dummy - Jeu d'apprentissage ===


Accuracy : 0.838
Precision : 0.000
Recall : 0.000

Matrice de confusion :
[[986   0]
 [190   0]]

Classification report :
              precision    recall  f1-score   support

         Non       0.84      1.00      0.91       986
         Oui       0.00      0.00      0.00       190

    accuracy                           0.84      1176
   macro avg       0.42      0.50      0.46      1176
weighted avg       0.70      0.84      0.76      1176



#### **Évaluation - jeu de teste**

In [10]:
# Évaluation du modèle Dummy sur le jeu de test

y_pred_dummy = dummy_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_dummy))
print("Precision :", precision_score(y_test, y_pred_dummy, pos_label="Oui", zero_division=0))
print("Recall :", recall_score(y_test, y_pred_dummy, pos_label="Oui", zero_division=0))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_dummy))

print("\nRapport de classification :")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

Accuracy : 0.8401360544217688
Precision : 0.0
Recall : 0.0

Matrice de confusion :
[[247   0]
 [ 47   0]]

Rapport de classification :
              precision    recall  f1-score   support

         Non       0.84      1.00      0.91       247
         Oui       0.00      0.00      0.00        47

    accuracy                           0.84       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.71      0.84      0.77       294



#### **Conclusions Dummy**

- > Le modèle **Dummy** obtient une accuracy de **83,8 %* sur le jeu d’apprentissage et de **84,0 %** sur le jeu de test, en prédisant systématiquement la classe majoritaire `Non`.
- > Cependant, il ne détecte aucun employé ayant quitté l’entreprise : la précision et le rappel pour la classe `Oui` sont donc de 0 %. Ce modèle constitue ainsi une référence minimale que les modèles suivants devront dépasser, notamment en améliorant la détection des départs et en réduisant les faux négatifs.

### **Standardisation des variables**

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled :", X_train_scaled.shape)
print("X_test_scaled  :", X_test_scaled.shape)

X_train_scaled : (1176, 45)
X_test_scaled  : (294, 45)


### **Modèle linéaire : Logistic Regression**

- > La régression logistique est un modèle linéaire adapté aux problèmes de classification binaire. Elle permet d'estimer la probabilité qu'un employé quitte l'entreprise à partir des variables explicatives. Ses performances seront comparées à celles du modèle Dummy afin d'évaluer sa capacité à détecter les départs.

In [12]:
# Modèle Logistic Regression

logreg_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logreg_model.fit(X_train_scaled, y_train)

print("Entraînement du modèle Logistic Regression OK")

Entraînement du modèle Logistic Regression OK


In [13]:
# Prédictions du modèle Logistic Regression

y_train_pred_logreg = logreg_model.predict(X_train_scaled)
y_test_pred_logreg = logreg_model.predict(X_test_scaled)

print("Prédictions Logistic Regression OK")

Prédictions Logistic Regression OK


#### **Évaluation - jeu d’apprentissage**

In [14]:
# Évaluation du modèle Logistic Regression

print("=== Logistic Regression - Jeu d'apprentissage ===")

print(f"Accuracy : {accuracy_score(y_train, y_train_pred_logreg):.3f}")
print(f"Precision : {precision_score(y_train, y_train_pred_logreg, pos_label='Oui', zero_division=0):.3f}")
print(f"Recall : {recall_score(y_train, y_train_pred_logreg, pos_label='Oui', zero_division=0):.3f}")

print("\nMatrice de confusion :")
print(confusion_matrix(y_train, y_train_pred_logreg, labels=["Non", "Oui"]))

print("\nClassification report :")
print(classification_report(
    y_train,
    y_train_pred_logreg,
    labels=["Non", "Oui"],
    zero_division=0
))

=== Logistic Regression - Jeu d'apprentissage ===
Accuracy : 0.895
Precision : 0.807
Recall : 0.463

Matrice de confusion :
[[965  21]
 [102  88]]

Classification report :
              precision    recall  f1-score   support

         Non       0.90      0.98      0.94       986
         Oui       0.81      0.46      0.59       190

    accuracy                           0.90      1176
   macro avg       0.86      0.72      0.76      1176
weighted avg       0.89      0.90      0.88      1176



#### **Évaluation - jeu de test**

In [15]:
# Évaluation du modèle Logistic Regression - Jeu de test

print("=== Logistic Regression - Jeu de test ===")

print(f"Accuracy : {accuracy_score(y_test, y_test_pred_logreg):.3f}")
print(f"Precision : {precision_score(y_test, y_test_pred_logreg, pos_label='Oui', zero_division=0):.3f}")
print(f"Recall : {recall_score(y_test, y_test_pred_logreg, pos_label='Oui', zero_division=0):.3f}")

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_test_pred_logreg, labels=["Non", "Oui"]))

print("\nClassification report :")
print(classification_report(
    y_test,
    y_test_pred_logreg,
    labels=["Non", "Oui"],
    zero_division=0
))

=== Logistic Regression - Jeu de test ===
Accuracy : 0.857
Precision : 0.593
Recall : 0.340

Matrice de confusion :
[[236  11]
 [ 31  16]]

Classification report :
              precision    recall  f1-score   support

         Non       0.88      0.96      0.92       247
         Oui       0.59      0.34      0.43        47

    accuracy                           0.86       294
   macro avg       0.74      0.65      0.68       294
weighted avg       0.84      0.86      0.84       294



#### **Conclusion du modèle Logistic Regression**

- > La régression logistique améliore les performances du modèle Dummy, avec une accuracy de 85,7 % sur le jeu de test, contre 84,0 % pour le Dummy. Elle permet également de détecter 34,0 % des employés ayant quitté l'entreprise, alors que le Dummy n'en détecte aucun. Cependant, l'écart entre les performances d'apprentissage et de test, notamment sur le rappel (46,3 % contre 34,0 %), indique un possible overfitting. Le modèle produit encore 31 faux négatifs sur le jeu de test.

### **Modèle non-linéaire : Random Forest**

- > Le Random Forest est un modèle non-linéaire basé sur un ensemble d'arbres de décision. Il permet de modéliser des relations plus complexes entre les variables explicatives et la variable cible. Ses performances seront comparées à celles du Dummy et de la régression logistique afin d'évaluer l'intérêt d'une approche non-linéaire.**

In [16]:
# Modèle Random Forest

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Entraînement du modèle Random Forest OK")

Entraînement du modèle Random Forest OK


In [17]:
# Prédictions du modèle Random Forest

y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

print("Prédictions Random Forest OK")

Prédictions Random Forest OK


#### **Évaluation - jeu d’apprentissage**

In [18]:
# Évaluation du modèle Random Forest - Jeu d'apprentissage

print("=== Random Forest - Jeu d'apprentissage ===")

print(f"Accuracy : {accuracy_score(y_train, y_train_pred_rf):.3f}")
print(f"Precision : {precision_score(y_train, y_train_pred_rf, pos_label='Oui', zero_division=0):.3f}")
print(f"Recall : {recall_score(y_train, y_train_pred_rf, pos_label='Oui', zero_division=0):.3f}")

print("\nMatrice de confusion :")
print(confusion_matrix(y_train, y_train_pred_rf, labels=["Non", "Oui"]))

print("\nClassification report :")
print(classification_report(
    y_train,
    y_train_pred_rf,
    labels=["Non", "Oui"],
    zero_division=0
))

=== Random Forest - Jeu d'apprentissage ===
Accuracy : 1.000
Precision : 1.000
Recall : 1.000

Matrice de confusion :
[[986   0]
 [  0 190]]

Classification report :
              precision    recall  f1-score   support

         Non       1.00      1.00      1.00       986
         Oui       1.00      1.00      1.00       190

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176



#### **Évaluation - jeu de teste**

In [19]:
# Évaluation du modèle Random Forest - Jeu de test

print("=== Random Forest - Jeu de test ===")

print(f"Accuracy : {accuracy_score(y_test, y_test_pred_rf):.3f}")
print(f"Precision : {precision_score(y_test, y_test_pred_rf, pos_label='Oui', zero_division=0):.3f}")
print(f"Recall : {recall_score(y_test, y_test_pred_rf, pos_label='Oui', zero_division=0):.3f}")

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_test_pred_rf, labels=["Non", "Oui"]))

print("\nClassification report :")
print(classification_report(
    y_test,
    y_test_pred_rf,
    labels=["Non", "Oui"],
    zero_division=0
))

=== Random Forest - Jeu de test ===
Accuracy : 0.857
Precision : 0.692
Recall : 0.191

Matrice de confusion :
[[243   4]
 [ 38   9]]

Classification report :
              precision    recall  f1-score   support

         Non       0.86      0.98      0.92       247
         Oui       0.69      0.19      0.30        47

    accuracy                           0.86       294
   macro avg       0.78      0.59      0.61       294
weighted avg       0.84      0.86      0.82       294



#### **Conclusion du modèle non-linéaire**

- > Le Random Forest présente un fort overfitting : 100 % d'accuracy sur le jeu d'apprentissage contre 85,4 % sur le jeu de test. Avec un rappel de 17,0 % pour la classe `Oui`, il détecte peu de départs réels

### **Comparaison des modèles**

In [20]:
# Comparaison des performances des modèles

comparaison_modeles = pd.DataFrame({
    "Modèle": [
        "Dummy",
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_test_pred_dummy),
        accuracy_score(y_test, y_test_pred_logreg),
        accuracy_score(y_test, y_test_pred_rf)
    ],
    "Precision": [
        precision_score(y_test, y_test_pred_dummy, pos_label="Oui", zero_division=0),
        precision_score(y_test, y_test_pred_logreg, pos_label="Oui", zero_division=0),
        precision_score(y_test, y_test_pred_rf, pos_label="Oui", zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_test_pred_dummy, pos_label="Oui", zero_division=0),
        recall_score(y_test, y_test_pred_logreg, pos_label="Oui", zero_division=0),
        recall_score(y_test, y_test_pred_rf, pos_label="Oui", zero_division=0)
    ],
    "F1-score": [
        f1_score(y_test, y_test_pred_dummy, pos_label="Oui", zero_division=0),
        f1_score(y_test, y_test_pred_logreg, pos_label="Oui", zero_division=0),
        f1_score(y_test, y_test_pred_rf, pos_label="Oui", zero_division=0)
    ]
})

print(comparaison_modeles.round(3))

                Modèle  Accuracy  Precision  Recall  F1-score
0                Dummy     0.840      0.000   0.000     0.000
1  Logistic Regression     0.857      0.593   0.340     0.432
2        Random Forest     0.857      0.692   0.191     0.300


- > La régression logistique obtient les meilleures performances globales sur le jeu de test, avec une accuracy de 85,7 % et un rappel de 34,0 % pour la classe `Oui`. Le Random Forest présente un fort overfitting, tandis que le modèle Dummy sert de référence avec une accuracy de 84,0 %.

### **Conclusion**

- > La régression logistique obtient les meilleures performances globales sur le jeu de test, avec une accuracy de 85,7 % et un rappel de 34,0 % pour la classe `Oui`. Le Random Forest présente un fort overfitting, tandis que le modèle Dummy sert de référence avec une accuracy de 84,0 %.